# Taxi Trip Revenue Analysis

A professional data analysis project using a real taxi trips dataset from Seaborn.

This project analyzes taxi trip revenue, tips, payment behavior, passenger count, and trip distance to support business decisions.

## Business Question

**How can a taxi service improve revenue and tipping performance based on trip behavior, payment method, and distance?**


## Project Goals

- Load a real-world taxi trips dataset directly from Seaborn
- Clean and prepare the data
- Create business KPIs
- Analyze fare, tip, distance, and passenger behavior
- Visualize key business patterns
- Provide practical business recommendations

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## Load Dataset

In [ ]:
# Load taxi dataset directly from Seaborn
df = sns.load_dataset('taxis')

df.head()

## Dataset Overview

In [ ]:
print("Dataset shape:", df.shape)
df.info()

In [ ]:
df.describe()

## Data Cleaning

In [ ]:
# Check missing values
print("Missing values:")
print(df.isnull().sum())

# Check duplicate rows
print("\nDuplicate rows:", df.duplicated().sum())

# Drop rows with missing important values
df = df.dropna(subset=['fare', 'tip', 'total', 'distance', 'payment'])

# Convert datetime columns
df['pickup'] = pd.to_datetime(df['pickup'])
df['dropoff'] = pd.to_datetime(df['dropoff'])

# Create new useful columns
df['trip_duration_min'] = (df['dropoff'] - df['pickup']).dt.total_seconds() / 60
df['pickup_hour'] = df['pickup'].dt.hour
df['pickup_day'] = df['pickup'].dt.day_name()
df['tip_percentage'] = df['tip'] / df['fare']

# Replace invalid values
df['tip_percentage'] = df['tip_percentage'].replace([np.inf, -np.inf], np.nan)

# Remove unrealistic/invalid rows
df = df[(df['fare'] > 0) & (df['total'] > 0) & (df['distance'] >= 0) & (df['trip_duration_min'] > 0)]

df.head()

**Cleaning Note:**  
Missing values were handled, datetime columns were converted, and new business metrics were created: trip duration, pickup hour, pickup day, and tip percentage.

## Business KPIs

In [ ]:
total_revenue = df['total'].sum()
total_fare = df['fare'].sum()
total_tips = df['tip'].sum()
average_trip_value = df['total'].mean()
average_tip = df['tip'].mean()
average_distance = df['distance'].mean()
average_tip_percentage = df['tip_percentage'].mean()

print("Total Revenue:", round(total_revenue, 2))
print("Total Fare:", round(total_fare, 2))
print("Total Tips:", round(total_tips, 2))
print("Average Trip Value:", round(average_trip_value, 2))
print("Average Tip:", round(average_tip, 2))
print("Average Distance:", round(average_distance, 2))
print("Average Tip Percentage:", round(average_tip_percentage * 100, 2), "%")

**Insight:**  
The KPIs summarize revenue, fare performance, tipping behavior, and average trip distance.

## 1. Revenue Distribution

In [ ]:
sns.set_style('whitegrid')

sns.histplot(
    data=df,
    x='total',
    kde=True
)

plt.title('Trip Revenue Distribution')
plt.xlabel('Total Trip Amount')
plt.ylabel('Count')
plt.show()

**Insight:**  
Most trips generate lower to moderate revenue, while only a smaller number of trips generate high revenue.

## 2. Fare vs Tip

In [ ]:
sns.scatterplot(
    data=df,
    x='fare',
    y='tip',
    hue='payment'
)

plt.title('Fare vs Tip by Payment Method')
plt.xlabel('Fare')
plt.ylabel('Tip')
plt.show()

**Insight:**  
Higher fares generally lead to higher tips. Payment method can also affect tipping behavior.

## 3. Total Revenue by Payment Method

In [ ]:
revenue_by_payment = df.groupby('payment', as_index=False)['total'].sum().sort_values(by='total', ascending=False)

sns.barplot(
    data=revenue_by_payment,
    x='payment',
    y='total'
)

plt.title('Total Revenue by Payment Method')
plt.xlabel('Payment Method')
plt.ylabel('Total Revenue')
plt.show()

**Insight:**  
Payment method analysis helps identify which customer payment behavior contributes more to total revenue.

## 4. Average Tip by Payment Method

In [ ]:
avg_tip_payment = df.groupby('payment', as_index=False)['tip'].mean().sort_values(by='tip', ascending=False)

sns.barplot(
    data=avg_tip_payment,
    x='payment',
    y='tip'
)

plt.title('Average Tip by Payment Method')
plt.xlabel('Payment Method')
plt.ylabel('Average Tip')
plt.show()

**Insight:**  
Average tip by payment method shows which payment type is associated with better tipping behavior.

## 5. Distance vs Total Revenue

In [ ]:
sns.scatterplot(
    data=df,
    x='distance',
    y='total',
    hue='payment'
)

plt.title('Distance vs Total Trip Amount')
plt.xlabel('Distance')
plt.ylabel('Total Trip Amount')
plt.show()

**Insight:**  
Longer trips usually generate higher total revenue, but outliers may exist depending on fare structure and fees.

## 6. Revenue by Pickup Hour

In [ ]:
hourly_revenue = df.groupby('pickup_hour', as_index=False)['total'].sum()

plt.figure(figsize=(10, 5))
sns.lineplot(
    data=hourly_revenue,
    x='pickup_hour',
    y='total',
    marker='o'
)

plt.title('Total Revenue by Pickup Hour')
plt.xlabel('Pickup Hour')
plt.ylabel('Total Revenue')
plt.show()

**Insight:**  
Hourly revenue helps identify peak demand times. This can support driver scheduling and operational planning.

## 7. Revenue by Pickup Day

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

daily_revenue = df.groupby('pickup_day', as_index=False)['total'].sum()
daily_revenue['pickup_day'] = pd.Categorical(daily_revenue['pickup_day'], categories=day_order, ordered=True)
daily_revenue = daily_revenue.sort_values('pickup_day')

plt.figure(figsize=(10, 5))
sns.barplot(
    data=daily_revenue,
    x='pickup_day',
    y='total'
)

plt.title('Total Revenue by Pickup Day')
plt.xlabel('Pickup Day')
plt.ylabel('Total Revenue')
plt.xticks(rotation=45)
plt.show()

**Insight:**  
Revenue by day shows which weekdays generate stronger taxi demand.

## 8. Passenger Count vs Total Revenue

In [ ]:
passenger_revenue = df.groupby('passengers', as_index=False)['total'].mean()

sns.barplot(
    data=passenger_revenue,
    x='passengers',
    y='total'
)

plt.title('Average Trip Revenue by Passenger Count')
plt.xlabel('Passenger Count')
plt.ylabel('Average Total Revenue')
plt.show()

**Insight:**  
Passenger count can help understand whether group rides are associated with higher average revenue.

## 9. Correlation Analysis

In [ ]:
correlation = df[['fare', 'tip', 'total', 'distance', 'trip_duration_min', 'passengers', 'tip_percentage']].corr()
correlation

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(
    correlation,
    annot=True
)

plt.title('Correlation Heatmap')
plt.show()

**Insight:**  
Correlation analysis shows how fare, tip, distance, duration, passengers, and total revenue relate to each other.

## 10. Trend Analysis: Fare vs Total Revenue

In [ ]:
sns.lmplot(
    data=df,
    x='fare',
    y='total',
    hue='payment',
    height=5,
    aspect=1.5
)

plt.title('Fare vs Total Revenue Trend')
plt.xlabel('Fare')
plt.ylabel('Total Revenue')
plt.show()

**Insight:**  
The trend confirms that fare is a major driver of total revenue. Improving trip volume during high-demand times can increase revenue.

## Final Business Recommendations

Based on the analysis:

1. **Focus driver availability during peak hours**  
   Revenue by pickup hour helps identify when demand is strongest.

2. **Encourage digital/card payments if they produce higher tips**  
   Payment behavior can be used to improve tipping performance.

3. **Prioritize longer-distance demand zones**  
   Longer trips usually generate higher revenue.

4. **Monitor low-tip segments**  
   Payment method and trip type can highlight where tip performance is weaker.

5. **Use weekday patterns for scheduling**  
   Pickup day analysis can improve workforce planning.

## Final Conclusion

This project shows how taxi trip data can support business decisions.  
Using Python, pandas, NumPy, matplotlib, and Seaborn, the analysis identifies revenue patterns, tipping behavior, payment impact, and operational opportunities.

## Tools Used

- Python
- pandas
- NumPy
- matplotlib
- seaborn

## README Text for GitHub

```markdown
# Taxi Trip Revenue Analysis

A business-focused data analysis project using a real taxi trips dataset.

## Business Question
How can a taxi service improve revenue and tipping performance based on trip behavior, payment method, and distance?

## Key Findings
- Higher fares generally lead to higher tips
- Payment method affects tipping behavior
- Longer trips usually generate higher revenue
- Revenue changes by pickup hour and pickup day
- Fare is a major driver of total trip revenue

## What I did
- Loaded a taxi dataset directly from Seaborn
- Cleaned missing values and invalid rows
- Created business KPIs
- Created new features like trip duration and tip percentage
- Analyzed revenue, tips, distance, passenger count, and payment method
- Used correlation and trend analysis
- Added business insights and recommendations

## Tools
- Python
- pandas
- NumPy
- matplotlib
- seaborn

## Purpose
This project demonstrates my ability to analyze business data, create KPIs, visualize performance, identify revenue drivers, and provide practical recommendations.
```
